In [13]:
import json
from pathlib import Path

import pandas as pd
import requests
from pydantic import BaseModel, ConfigDict, Field, ValidationError


In [14]:
url = (
    r"https://www.ncei.noaa.gov/erddap/tabledap/deep_sea_corals.jsonlKVP"
    r"?CatalogNumber,ScientificName,latitude,longitude,DepthInMeters,LargeMarineEcosystem,Country,ObservationDate"
    r"&ShallowFlag=0"
    r"&ObservationYear%3E=2020"
    r"&Country=%22USA%22"
    r"&LargeMarineEcosystem=~%22%28Southeast%20U.S.%20Continental%20Shelf%7CGulf%20of%20Mexico%7CCaribbean%20Sea%29%22"
)

In [15]:
class DeepSeaCoralRecord(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    latitude: float
    longitude: float
    catalog_num: int = Field(alias="CatalogNumber")
    scientific_name: str | None = Field(default=None, alias="ScientificName")
    depth_m: float | None = Field(default=None, alias="DepthInMeters")
    large_marine_ecosystem: str = Field(alias="LargeMarineEcosystem")
    country: str = Field(alias="Country")
    observation_date: str = Field(alias="ObservationDate")

In [16]:
def download(url: str, dest: Path) -> None:
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            f.writelines(r.iter_content(1 << 20))

In [17]:
def read_records(file_path: Path) -> list[DeepSeaCoralRecord]:
    records: list[DeepSeaCoralRecord] = []
    with open(file_path) as f:
        for line in f:
            try:
                record = DeepSeaCoralRecord(**json.loads(line))
                records.append(record)
            except ValidationError:
                continue
    return records

In [18]:
dest = Path("../data/raw/deep_sea_corals_se.jsonl")
dest.parent.mkdir(exist_ok=True)

download(url, dest)
records = read_records(dest)

print(f"Validated: {len(records)} records")

Validated: 60169 records


In [19]:
df = pd.DataFrame(record.model_dump() for record in records)
df.head(10)

,latitude,longitude,catalog_num,scientific_name,depth_m,large_marine_ecosystem,country,observation_date
0,17.7767,-64.8705,1064877,Swiftia exserta,93.0,Caribbean Sea,USA,2020-07-06
1,17.7768,-64.8705,1064878,Swiftia exserta,93.0,Caribbean Sea,USA,2020-07-06
2,17.7768,-64.8705,1064879,Nicella,93.0,Caribbean Sea,USA,2020-07-06
3,17.7767,-64.8705,1064880,Swiftia exserta,84.0,Caribbean Sea,USA,2020-07-06
4,17.7767,-64.8705,1064881,Nicella,84.0,Caribbean Sea,USA,2020-07-06
5,17.7767,-64.8705,1064882,Swiftia exserta,84.0,Caribbean Sea,USA,2020-07-06
6,17.7767,-64.8705,1064883,Swiftia exserta,84.0,Caribbean Sea,USA,2020-07-06
7,17.7767,-64.8705,1064884,Antipatharia,84.0,Caribbean Sea,USA,2020-07-06
8,17.7766,-64.8706,1064885,Swiftia exserta,86.0,Caribbean Sea,USA,2020-07-06
9,17.7766,-64.8706,1064886,Plexauridae,85.0,Caribbean Sea,USA,2020-07-06
